In [1]:
# IMPORTS

from classes.single_encoder import SingleEncoder
from classes.dual_encoder_single import DualEncoderAsSingle
from helpers.generate_embeddings import generate_embeddings
from helpers.load_embeddings import load_embeddings
from helpers.retrieve_top_k import retrieve_top_k
from helpers.load_dual_encoder import load_dual_encoder_model
from helpers.load_single_encoder import load_single_encoder_model
from helpers.retrieve_weight_samples import retrieve_weighted_samples
from transformers import CanineModel, CanineTokenizer
import pandas as pd
import torch
import faiss
import numpy as np
import pickle
import os
from openai import OpenAI

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Skipping import of cpp extensions due to incompatible torch version 2.7.1+cu118 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0119 12:30:22.257000 23764 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [16]:
# SETUP
API_KEY = ""
with open("API_KEY", "r") as f:
    API_KEY = f.read()
client = OpenAI(api_key=API_KEY)

MODEL_PATH = "../output/models/"
EMBEDDINGS_PATH = "../output/embeddings/"
CHAT_DATA_PATH = "../data/rag/"

MODEL_NAME = "base_canine"
# MODEL_NAME = "author_contrastive"
# MODEL_NAME = "style_discovery_loss"

CHAT_DATA = "private_embeddings_full"


MODEL_PATH = MODEL_PATH + MODEL_NAME
EMBEDDINGS_PATH = EMBEDDINGS_PATH + MODEL_NAME
CHAT_DATA_PATH = CHAT_DATA_PATH + CHAT_DATA + ".csv"

#=====================================
# LOAD MODEL
# Base CANINE
tokenizer = CanineTokenizer.from_pretrained("google/canine-s")
encoder = CanineModel.from_pretrained("google/canine-s")
model = SingleEncoder()
model.encoder = encoder  # replace the encoder
model.to(device)
model.eval()

# Author Contrastive Loss
# model, tokenizer, device = load_single_encoder_model(MODEL_PATH)

# Style Discovery Loss
# model, tokenizer, device = load_dual_encoder_model(MODEL_PATH)
# model = DualEncoderAsSingle(model)

SingleEncoder(
  (encoder): CanineModel(
    (char_embeddings): CanineEmbeddings(
      (HashBucketCodepointEmbedder_0): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_1): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_2): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_3): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_4): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_5): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_6): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_7): Embedding(16384, 96)
      (char_position_embeddings): Embedding(16384, 768)
      (token_type_embeddings): Embedding(16, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (initial_char_encoder): CanineEncoder(
      (layer): ModuleList(
        (0): CanineLayer(
          (attention): CanineAttention(
            (self): CanineSelfAttention(
              (query): Linear

In [17]:
# LOAD CHAT DATA AND GENERATE EMBEDDINGS
df = pd.read_csv(CHAT_DATA_PATH)
df = df.dropna(subset=["Content"])
df["Content"] = df["Content"].astype(str)

print("Messages:", len(df))
print("Authors:", df["Author"].nunique())

texts = df["Content"].tolist()

embeddings = generate_embeddings(
    texts=texts,
    model=model,
    tokenizer=tokenizer,
    batch_size=8,
    device=device,
)

print("Embedding shape:", embeddings.shape)

records = [
    {
        "author": df.iloc[i]["Author"],
        "content": df.iloc[i]["Content"]
    }
    for i in range(len(df))
]

# Save to FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # cosine similarity
index.add(embeddings.numpy())

print("Vectors in index:", index.ntotal)

os.makedirs(EMBEDDINGS_PATH, exist_ok=True)

faiss.write_index(index, os.path.join(EMBEDDINGS_PATH, "style_index.faiss"))
with open(os.path.join(EMBEDDINGS_PATH, "style_metadata.pkl"), "wb") as f:
    pickle.dump(records, f)
torch.save(embeddings, os.path.join(EMBEDDINGS_PATH, "style_embeddings.pt"))

Messages: 40713
Authors: 2


100%|██████████| 5090/5090 [01:53<00:00, 44.88it/s]


Embedding shape: torch.Size([40713, 128])
Vectors in index: 40713


In [4]:
# load embeddings directly if already generated
index, records, embeddings = load_embeddings(EMBEDDINGS_PATH)

In [ ]:
# Example query
query_text = "what are you doing my brother in christ"

# Retrieve weighted samples
retrieved_results = retrieve_weighted_samples(
    query_text=query_text,
    model=model,
    tokenizer=tokenizer,
    index=index,
    records=records,
    device=device,
    k=20,
    temperature=0.03  # Adjust for more/less diversity
)

# Print results
for i, result in enumerate(retrieved_results):
    print(f"\n--- Result {i+1} ---")
    print(f"Author: {result['author']}")
    print(f"Content: {result['content']}")
    print(f"Probability Rank: {result['rank_by_probability']}")

In [18]:
# Generate response using OpenAI GPT model

def build_prompt(
    user_message,
    style_examples
):
    style_block = "\n".join(f"- {ex['content']}" for ex in style_examples)

    prompt = f"""You will receive a message as part of a text conversation.
You are going to give a reply suggestion based on the writing style inferred from provided examples.

Based on the examples below, infer the writing style, such as tone, formality, common phrases, and slang.
Do NOT reuse topics, facts, names, or specific phrases from the examples.

Examples:
{style_block}

Write a short, 1-clause, sentence, non-specific reply to the message, using ONLY the inferred style from the examples above.
Do not infer style from the message itself.
Do not make up any context in your response.
Do not add extra explanation or context outside of the response.
"""
    return prompt

# def build_prompt(
#     user_message,
#     style_examples
# ):
#     style_block = "\n".join(f"- {ex['content']}" for ex in style_examples)

#     prompt = f"""You will receive a message as part of a text conversation.
# You are going to give a reply suggestion to this message.

# Write a short, 1-clause, sentence, non-specific reply to the message.
# Ensure the reply is stylistically similar to the message.
# Do not make up any context in your response.
# Do not add extra explanation or context outside of the response.
# """
#     return prompt


def generate_responses(query_text, model_name, model, tokenizer, device, client, k=3, gpt_model="gpt-5-mini"):
    index, records, embeddings = load_embeddings("../output/embeddings/" + model_name)
    responses = []

    for i in range(k):
        retrieved_results = retrieve_weighted_samples(
            query_text=query_text,
            model=model,
            tokenizer=tokenizer,
            index=index,
            records=records,
            device=device,
            k=10,
            temperature=0.03  # Adjust for more/less diversity
        )
        
        response = client.responses.create(
            model=gpt_model,
            instructions=build_prompt(
                user_message=query_text,
                style_examples=retrieved_results
            ),
            input=query_text,
        )

        responses.append(response.output_text)
        
    return responses

In [ ]:
from helpers.generate_messages_with_context import message_to_string_format
import json

with open(r'..\output\rag\selected_messages_with_context.json', 'r') as f:
    data = json.load(f)

messages = [
    "when's gaming?",
    "yo wsg",
    "@grok is this true?",
    "swag nation we rise",
    "ok buddy",
    "whole milk or full milk?",
]

for m in messages:
    print(generate_responses(m, MODEL_NAME, model, tokenizer, device, client, k=1)[0])
# for m in data["selected_conversations"]:
#     print(generate_responses(m["current_message"]["Content"], MODEL_NAME, model, tokenizer, device, client, k=1)[0])


when the caffeine kicks in
how's everything going?
you'll figure it out
you good tho?
i'd double-check that.
hell yeah we got this.
ok cool lemme know bro
where the charger at
what platform tho
this is peak chaos????
i cant handle this rn
how you planning to make that work
i havent looked into that
same energy bro
i'm down with that vibe
chill out bro
bruh that's fucked up
this is wild!!!!
does that even make sense bro
thats cool
